# Commercial Model Comparison (Thesis Section 4.5)

Compares 6 models on the test set (510 samples):
- **Gemma 2 9B-IT (Zero-Shot)**
- **Gemma 2 9B-IT (Fine-Tuned)**
- **GPT-4o**
- **GPT-5.2**
- **Claude Opus 4.6**
- **Claude Sonnet 4**

**Sections:**
1. Load All Model Results
2. Overall Comparison Table
3. Per-Rubric QWK and MAE
4. Off-Topic Detection Comparison
5. Paired Bootstrap Significance Tests
6. Per-Question QWK Comparison
7. Per-Question MAE Comparison
8. Cost Analysis

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score

sns.set_theme(style='whitegrid', font_scale=1.1)

RUBRICS = ['Clarity', 'Terminology', 'Coverage', 'Accuracy']

## 1. Load All Model Results

In [ ]:
model_paths = {
    'Gemma ZS': '../Finetuning/zeroshot_finetuned_prompt_results.csv',
    'Gemma FT': '../Finetuning/finetuned_results.csv',
    'GPT-4o': '../GPT_Comparison/gpt-4o/test_results.csv',
    'GPT-5.2': '../GPT_Comparison/gpt-5_2/test_results.csv',
    'Claude Opus 4.6': '../Claude_Comparison/claude_opus_4_6/test_results.csv',
    'Claude Sonnet 4': '../Claude_Comparison/claude_sonnet_4_20250514/test_results.csv'
}

models = {}
for name, path in model_paths.items():
    df = pd.read_csv(path)
    # Normalize column names
    col_map = {}
    for c in df.columns:
        if c in ('Within_1', 'Within_1.0'): col_map[c] = 'Within_1'
        elif c in ('Within_05', 'Within_0.5'): col_map[c] = 'Within_05'
    df = df.rename(columns=col_map)

    # Ensure Human_Total and LLM_Total exist
    if 'Human_Total' not in df.columns:
        df['Human_Total'] = df['Human_Clarity'] + df['Human_Terminology'] + df['Human_Coverage'] + df['Human_Accuracy']
    if 'LLM_Total' not in df.columns:
        df['LLM_Total'] = df['LLM_Clarity'] + df['LLM_Terminology'] + df['LLM_Coverage'] + df['LLM_Accuracy']

    # Ensure Human_Final / LLM_Final
    if 'Human_Final' not in df.columns:
        df['Human_Final'] = df['Human_Total'] / 2
    if 'LLM_Final' not in df.columns:
        df['LLM_Final'] = df['LLM_Total'] / 2

    # Extract Q_num from student_id
    if 'Q_num' not in df.columns:
        id_col = 'student_id' if 'student_id' in df.columns else df.columns[0]
        df['Q_num'] = df[id_col].astype(str).str.extract(r'Q(\d+)')[0].astype(int)

    models[name] = df
    print(f'{name:20s}: {len(df)} samples loaded')

## 2. Overall Comparison Table

In [ ]:
def compute_metrics(df):
    diff = np.abs(df['LLM_Final'].values - df['Human_Final'].values)
    acc_1 = np.mean(diff <= 1.0) * 100
    acc_05 = np.mean(diff <= 0.5) * 100
    mae = np.mean(diff)

    h_total = df['Human_Total'].astype(int).values
    l_total = np.clip(df['LLM_Total'].astype(int).values, 0, 10)
    qwk = cohen_kappa_score(h_total, l_total, weights='quadratic')

    # Off-topic accuracy
    if 'OffTopic_Match' in df.columns:
        ot_acc = df['OffTopic_Match'].astype(bool).mean() * 100
    else:
        ot_acc = np.nan

    return {'Acc ±1.0 (%)': acc_1, 'Acc ±0.5 (%)': acc_05, 'MAE': mae, 'QWK': qwk, 'OT Acc (%)': ot_acc}


rows = []
for name, df in models.items():
    m = compute_metrics(df)
    m['Model'] = name
    rows.append(m)

comp_df = pd.DataFrame(rows)[['Model', 'Acc ±1.0 (%)', 'Acc ±0.5 (%)', 'MAE', 'QWK', 'OT Acc (%)']]
comp_df = comp_df.round(3)
print('Overall Model Comparison (510 test samples):')
comp_df

In [ ]:
# Grouped bar chart
fig, axes = plt.subplots(1, 4, figsize=(20, 6))
model_names = list(models.keys())
colors = ['#95a5a6', '#2ecc71', '#3498db', '#2980b9', '#e74c3c', '#c0392b']

for ax, metric in zip(axes, ['Acc ±1.0 (%)', 'Acc ±0.5 (%)', 'MAE', 'QWK']):
    values = [compute_metrics(models[m])[metric] for m in model_names]
    bars = ax.bar(range(len(model_names)), values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(model_names)))
    ax.set_xticklabels([n.replace(' ', '\n') for n in model_names], fontsize=8)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    for bar, val in zip(bars, values):
        fmt = f'{val:.1f}' if '%' in metric else f'{val:.3f}'
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height(),
                fmt, ha='center', va='bottom', fontsize=8)

fig.suptitle('6-Model Performance Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Per-Rubric QWK and MAE

In [ ]:
def per_rubric_qwk_mae(df):
    results = {}
    for rubric in RUBRICS:
        h = df[f'Human_{rubric}'].values.astype(int)
        l = df[f'LLM_{rubric}'].values.astype(int)
        max_score = 4 if rubric == 'Accuracy' else 2
        l = np.clip(l, 0, max_score)
        qwk = cohen_kappa_score(h, l, weights='quadratic')
        mae = np.mean(np.abs(h - l))
        results[rubric] = {'QWK': qwk, 'MAE': mae}
    return results


# QWK comparison
qwk_rows = []
for name, df in models.items():
    rubric_metrics = per_rubric_qwk_mae(df)
    row = {'Model': name}
    for rubric in RUBRICS:
        row[f'{rubric} QWK'] = round(rubric_metrics[rubric]['QWK'], 3)
    qwk_rows.append(row)

print('Per-Rubric QWK:')
display(pd.DataFrame(qwk_rows))

# MAE comparison
mae_rows = []
for name, df in models.items():
    rubric_metrics = per_rubric_qwk_mae(df)
    row = {'Model': name}
    for rubric in RUBRICS:
        row[f'{rubric} MAE'] = round(rubric_metrics[rubric]['MAE'], 3)
    mae_rows.append(row)

print('\nPer-Rubric MAE:')
pd.DataFrame(mae_rows)

In [ ]:
# Heatmap of QWK by model and rubric
qwk_matrix = np.zeros((len(models), len(RUBRICS)))
for i, (name, df) in enumerate(models.items()):
    rubric_metrics = per_rubric_qwk_mae(df)
    for j, rubric in enumerate(RUBRICS):
        qwk_matrix[i, j] = rubric_metrics[rubric]['QWK']

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(qwk_matrix, annot=True, fmt='.3f', cmap='RdYlGn',
            xticklabels=RUBRICS, yticklabels=list(models.keys()),
            vmin=0, vmax=1, ax=ax)
ax.set_title('Per-Rubric QWK by Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Off-Topic Detection Comparison

In [ ]:
def off_topic_metrics(df):
    if 'Human_OffTopic' in df.columns:
        human_ot = df['Human_OffTopic'].isin(['Off-Topic', 'No Answer'])
        llm_ot = df['LLM_OffTopic'].isin(['Off-Topic', 'No Answer'])
    elif 'Human_Off_Topic' in df.columns:
        human_ot = df['Human_Off_Topic'].astype(bool)
        llm_ot = df['LLM_Off_Topic'].astype(bool)
    else:
        return {'Precision': np.nan, 'Recall': np.nan, 'F1': np.nan, 'Accuracy (%)': np.nan}

    tp = (human_ot & llm_ot).sum()
    fp = (~human_ot & llm_ot).sum()
    fn = (human_ot & ~llm_ot).sum()
    tn = (~human_ot & ~llm_ot).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    acc = (tp + tn) / len(df) * 100

    return {'FP': int(fp), 'FN': int(fn), 'Precision': precision,
            'Recall': recall, 'F1': f1, 'Accuracy (%)': acc}


ot_rows = []
for name, df in models.items():
    m = off_topic_metrics(df)
    m['Model'] = name
    ot_rows.append(m)

ot_df = pd.DataFrame(ot_rows)
ot_df = ot_df[['Model', 'FP', 'FN', 'Precision', 'Recall', 'F1', 'Accuracy (%)']].round(3)
print('Off-Topic Detection Comparison:')
ot_df

## 5. Paired Bootstrap Significance Tests

In [ ]:
def bootstrap_comparison(df_base, df_other, n_bootstrap=10000, seed=42):
    """Paired bootstrap test: is base significantly better than other?"""
    rng = np.random.RandomState(seed)
    n = len(df_base)

    base_diff = np.abs(df_base['LLM_Final'].values - df_base['Human_Final'].values)
    other_diff = np.abs(df_other['LLM_Final'].values - df_other['Human_Final'].values)

    boot_acc1_diff = np.zeros(n_bootstrap)
    boot_mae_diff = np.zeros(n_bootstrap)
    boot_qwk_diff = np.zeros(n_bootstrap)

    for b in range(n_bootstrap):
        idx = rng.randint(0, n, size=n)
        b_base = base_diff[idx]
        b_other = other_diff[idx]

        boot_acc1_diff[b] = np.mean(b_base <= 1.0) - np.mean(b_other <= 1.0)
        boot_mae_diff[b] = np.mean(b_other) - np.mean(b_base)  # positive = base better

        h_total = df_base['Human_Total'].values[idx].astype(int)
        l_base_total = np.clip(df_base['LLM_Total'].values[idx].astype(int), 0, 10)
        l_other_total = np.clip(df_other['LLM_Total'].values[idx].astype(int), 0, 10)
        try:
            qwk_base = cohen_kappa_score(h_total, l_base_total, weights='quadratic')
            qwk_other = cohen_kappa_score(h_total, l_other_total, weights='quadratic')
            boot_qwk_diff[b] = qwk_base - qwk_other
        except:
            boot_qwk_diff[b] = 0

    return {
        'Acc±1 p': np.mean(boot_acc1_diff <= 0),
        'MAE p': np.mean(boot_mae_diff <= 0),
        'QWK p': np.mean(boot_qwk_diff <= 0)
    }


# Compare FT Gemma vs each other model
base_df = models['Gemma FT']
sig_rows = []

for name, df in models.items():
    if name == 'Gemma FT':
        continue
    print(f'  Testing Gemma FT vs {name}...')
    result = bootstrap_comparison(base_df, df)
    result['vs Model'] = name
    for k in ['Acc±1 p', 'MAE p', 'QWK p']:
        p = result[k]
        result[f'{k} sig'] = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'
    sig_rows.append(result)

sig_df = pd.DataFrame(sig_rows)
sig_df = sig_df[['vs Model', 'Acc±1 p', 'Acc±1 p sig', 'MAE p', 'MAE p sig', 'QWK p', 'QWK p sig']]
print('\nPaired Bootstrap Significance (FT Gemma vs each model, B=10,000):')
sig_df.round(4)

## 6. Per-Question QWK Comparison

In [ ]:
def per_question_qwk(df):
    results = {}
    for q in sorted(df['Q_num'].unique()):
        mask = df['Q_num'] == q
        h = df.loc[mask, 'Human_Total'].astype(int).values
        l = np.clip(df.loc[mask, 'LLM_Total'].astype(int).values, 0, 10)
        if len(np.unique(h)) > 1 and len(np.unique(l)) > 1:
            results[f'Q{q}'] = cohen_kappa_score(h, l, weights='quadratic')
        else:
            results[f'Q{q}'] = np.nan
    return results


q_qwk_rows = []
for name, df in models.items():
    row = per_question_qwk(df)
    row['Model'] = name
    q_qwk_rows.append(row)

q_qwk_df = pd.DataFrame(q_qwk_rows).set_index('Model').round(3)
print('Per-Question QWK:')
q_qwk_df

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(q_qwk_df.astype(float), annot=True, fmt='.2f', cmap='RdYlGn',
            vmin=0, vmax=1, ax=ax, linewidths=0.5)
ax.set_title('Per-Question QWK by Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Per-Question MAE Comparison

In [ ]:
def per_question_mae(df):
    results = {}
    for q in sorted(df['Q_num'].unique()):
        mask = df['Q_num'] == q
        diff = np.abs(df.loc[mask, 'LLM_Final'].values - df.loc[mask, 'Human_Final'].values)
        results[f'Q{q}'] = np.mean(diff)
    return results


q_mae_rows = []
for name, df in models.items():
    row = per_question_mae(df)
    row['Model'] = name
    q_mae_rows.append(row)

q_mae_df = pd.DataFrame(q_mae_rows).set_index('Model').round(3)
print('Per-Question MAE:')
q_mae_df

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(q_mae_df.astype(float), annot=True, fmt='.2f', cmap='RdYlGn_r',
            ax=ax, linewidths=0.5)
ax.set_title('Per-Question MAE by Model (lower is better)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Cost Analysis

In [ ]:
# Estimated costs per 510 test samples
# Based on approximate token usage: ~500 input tokens + ~100 output tokens per sample
cost_data = {
    'Model': ['Gemma FT (Self-Hosted)', 'GPT-4o', 'GPT-5.2', 'Claude Opus 4.6', 'Claude Sonnet 4'],
    'Input $/1M tok': [0, 2.50, 10.00, 15.00, 3.00],
    'Output $/1M tok': [0, 10.00, 30.00, 75.00, 15.00],
    'Est. Cost (510 samples)': [0, 0, 0, 0, 0],  # will compute below
    'Notes': [
        'One-time fine-tuning cost + free inference on local GPU',
        'OpenAI API pricing',
        'OpenAI API pricing',
        'Anthropic API pricing',
        'Anthropic API pricing'
    ]
}

# Compute estimated costs (approximate)
avg_input_tokens = 500
avg_output_tokens = 100
n_samples = 510

for i in range(len(cost_data['Model'])):
    input_cost = (avg_input_tokens * n_samples / 1_000_000) * cost_data['Input $/1M tok'][i]
    output_cost = (avg_output_tokens * n_samples / 1_000_000) * cost_data['Output $/1M tok'][i]
    cost_data['Est. Cost (510 samples)'][i] = round(input_cost + output_cost, 2)

cost_df = pd.DataFrame(cost_data)
print('Cost Comparison (estimated):')
cost_df

In [ ]:
# Performance vs cost scatter
fig, ax = plt.subplots(figsize=(10, 6))

model_order = ['Gemma FT', 'GPT-4o', 'GPT-5.2', 'Claude Opus 4.6', 'Claude Sonnet 4']
plot_colors = ['#2ecc71', '#3498db', '#2980b9', '#e74c3c', '#c0392b']

for name, color in zip(model_order, plot_colors):
    if name in models:
        m = compute_metrics(models[name])
        idx = cost_data['Model'].index(name if name != 'Gemma FT' else 'Gemma FT (Self-Hosted)')
        cost = cost_data['Est. Cost (510 samples)'][idx]
        ax.scatter(cost, m['QWK'], s=200, color=color, zorder=5, edgecolors='black')
        ax.annotate(name, (cost, m['QWK']), textcoords='offset points',
                    xytext=(10, 5), fontsize=10)

ax.set_xlabel('Estimated Cost ($) for 510 Samples', fontsize=12)
ax.set_ylabel('QWK', fontsize=12)
ax.set_title('Performance vs Cost Trade-off', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()